In [1]:
# Import the libraries used for data handling and load the dataset
import pandas as pd
import numpy as np

shopper_df = pd.read_csv("online_shoppers_intention.csv")
print(shopper_df.shape)

(12330, 18)


In [2]:
# Preview the first rows and check column types and missing values
shopper_df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [3]:
shopper_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [4]:
# Check data types, missing values and the balance of the target column
print(shopper_df.dtypes)
print(shopper_df.isnull().sum())
print(shopper_df["Revenue"].value_counts())

Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                       object
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                 object
Weekend                       bool
Revenue                       bool
dtype: object
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser 

In [5]:
# Inspect the distinct values of the four non-numeric columns before encoding them
for col_name in ["Month", "VisitorType", "Weekend", "Revenue"]:
    print(col_name, "|", shopper_df[col_name].dtype, "|", shopper_df[col_name].unique())

Month | object | ['Feb' 'Mar' 'May' 'Oct' 'June' 'Jul' 'Aug' 'Nov' 'Sep' 'Dec']
VisitorType | object | ['Returning_Visitor' 'New_Visitor' 'Other']
Weekend | bool | [False  True]
Revenue | bool | [False  True]


In [6]:
# Separate the input features from the target label
target_colm = shopper_df["Revenue"].astype(int)
features_df = shopper_df.drop(columns=["Revenue"])
print(features_df.shape, target_colm.shape)

(12330, 17) (12330,)


In [7]:
# Split into training and test sets, keeping the same buyer/non-buyer ratio in both
from sklearn.model_selection import train_test_split

training_X, testing_X, training_y, testing_y = train_test_split(
    features_df, target_colm, test_size=0.2, random_state=49, stratify=target_colm
)
print(training_X.shape, testing_X.shape)
print(training_y.mean(), testing_y.mean())

(9864, 17) (2466, 17)
0.15470397404703973 0.1549067315490673


In [8]:
# Group the columns by the preprocessing each will receive
catg_cols = ["Month", "VisitorType", "Weekend", "OperatingSystems", "Browser", "Region", "TrafficType"]
numr_cols = [c for c in training_X.columns if c not in catg_cols]
print(len(catg_cols), len(numr_cols))
print(numr_cols)

7 10
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']


In [9]:
# Build the preprocessor: scale numeric columns, one-hot encode categorical columns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

prepare_step = ColumnTransformer([
    ("scale_num", StandardScaler(), numr_cols),
    ("encode_cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), catg_cols),
])
prepare_step.fit(training_X)
print(prepare_step.transform(training_X).shape)

(9864, 75)


In [10]:
# Combine the preprocessing step with each of the five classifiers and train them
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

classifier_dict = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=45),
    "Decision Tree": DecisionTreeClassifier(random_state=59),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(probability=True, random_state=49),
}

fitted_pipelines = {}
for model_label, clf_obj in classifier_dict.items():
    fitted_pipelines[model_label] = Pipeline([
        ("prep", clone(prepare_step)),
        ("clf", clf_obj),
    ]).fit(training_X, training_y)
    print(model_label, "trained")

Logistic Regression trained
Decision Tree trained
KNN trained
Naive Bayes trained
Support Vector Machine trained


In [11]:
# Score each trained pipeline on the held-out test set and collect the six metrics in one table
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

buyer_scorecard_rows = []
for pipe_tag, trained_unit in fitted_pipelines.items():
    guess_labels = trained_unit.predict(testing_X)
    guess_buyer_prob = trained_unit.predict_proba(testing_X)[:, 1]
    buyer_scorecard_rows.append({
        "ML Model Name": pipe_tag,
        "Accuracy": accuracy_score(testing_y, guess_labels),
        "AUC": roc_auc_score(testing_y, guess_buyer_prob),
        "Precision": precision_score(testing_y, guess_labels),
        "Recall": recall_score(testing_y, guess_labels),
        "F1": f1_score(testing_y, guess_labels),
        "MCC": matthews_corrcoef(testing_y, guess_labels),
    })

buyer_metric_table = pd.DataFrame(buyer_scorecard_rows).round(4)
buyer_metric_table

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.8840,0.8866,0.7353,0.3927,0.5119,0.4817
1,Decision Tree,0.8658,0.7271,0.5726,0.5262,0.5484,0.4703
2,KNN,0.8767,0.8026,0.6696,0.4031,0.5033,0.4562
3,Naive Bayes,0.2786,0.7692,0.1753,0.9869,0.2977,0.1471
4,Support Vector Machine,0.8954,0.8739,0.7366,0.5052,0.5994,0.5543


In [12]:
# Show the confusion matrix of every model on the test set
from sklearn.metrics import confusion_matrix

for pipe_tag, trained_unit in fitted_pipelines.items():
    grid_counts = confusion_matrix(testing_y, trained_unit.predict(testing_X))
    print(pipe_tag)
    print(pd.DataFrame(
        grid_counts,
        index=["Actual: no buy", "Actual: buy"],
        columns=["Pred: no buy", "Pred: buy"],
    ))
    print()

Logistic Regression
                Pred: no buy  Pred: buy
Actual: no buy          2030         54
Actual: buy              232        150

Decision Tree
                Pred: no buy  Pred: buy
Actual: no buy          1934        150
Actual: buy              181        201

KNN
                Pred: no buy  Pred: buy
Actual: no buy          2008         76
Actual: buy              228        154

Naive Bayes
                Pred: no buy  Pred: buy
Actual: no buy           310       1774
Actual: buy                5        377

Support Vector Machine
                Pred: no buy  Pred: buy
Actual: no buy          2015         69
Actual: buy              189        193



In [13]:
# Build a separate Multinomial Naive Bayes pipeline with non-negative scaling and score it on the test set
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import MultinomialNB

multinom_prep_unit = ColumnTransformer([
    ("squeeze_num", MinMaxScaler(clip=True), numr_cols),
    ("onehot_cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), catg_cols),
])
multinom_pipeline = Pipeline([
    ("prep", multinom_prep_unit),
    ("clf", MultinomialNB()),
]).fit(training_X, training_y)

mnb_labels = multinom_pipeline.predict(testing_X)
mnb_buyer_prob = multinom_pipeline.predict_proba(testing_X)[:, 1]

mnb_metric_dict = {
    "Accuracy": accuracy_score(testing_y, mnb_labels),
    "AUC": roc_auc_score(testing_y, mnb_buyer_prob),
    "Precision": precision_score(testing_y, mnb_labels),
    "Recall": recall_score(testing_y, mnb_labels),
    "F1": f1_score(testing_y, mnb_labels),
    "MCC": matthews_corrcoef(testing_y, mnb_labels),
}
print({metric_key: round(metric_val, 4) for metric_key, metric_val in mnb_metric_dict.items()})
print(confusion_matrix(testing_y, mnb_labels))

{'Accuracy': 0.8398, 'AUC': 0.7529, 'Precision': 0.433, 'Recall': 0.1099, 'F1': 0.1754, 'MCC': 0.1555}
[[2029   55]
 [ 340   42]]


In [14]:
# Save every trained pipeline into a model folder so the Streamlit app can load them later
import os
import joblib

os.makedirs("model", exist_ok=True)
saved_file_map = {}
for pipe_tag, trained_unit in fitted_pipelines.items():
    file_stub = pipe_tag.lower().replace(" ", "_")
    saved_file_map[pipe_tag] = f"model/{file_stub}.joblib"
    joblib.dump(trained_unit, saved_file_map[pipe_tag])
print(saved_file_map)

{'Logistic Regression': 'model/logistic_regression.joblib', 'Decision Tree': 'model/decision_tree.joblib', 'KNN': 'model/knn.joblib', 'Naive Bayes': 'model/naive_bayes.joblib', 'Support Vector Machine': 'model/support_vector_machine.joblib'}


In [15]:
# Export the test split as test_data.csv, then reload it and a saved model to confirm the app's workflow works
test_export_frame = testing_X.copy()
test_export_frame["Revenue"] = testing_y
test_export_frame.to_csv("test_data.csv", index=False)

csv_roundtrip_frame = pd.read_csv("test_data.csv")
print(csv_roundtrip_frame.shape)

svm_from_disk = joblib.load(saved_file_map["Support Vector Machine"])
csv_input_part = csv_roundtrip_frame.drop(columns=["Revenue"])
csv_label_part = csv_roundtrip_frame["Revenue"]
print(round(accuracy_score(csv_label_part, svm_from_disk.predict(csv_input_part)), 4))

(2466, 18)
0.8954
